# 04 — Churn Prediction & Risk Stratification

*Time-aware binary classification: behavioural features → churn probability scores → three-tier retention targeting*

## Executive Summary

Builds a supervised churn model on the 7,671-customer base from NB01. The core design principle is **strict temporal isolation**: features are computed from pre-cutoff transactions; the churn label is observed in the 180-day window that follows. This eliminates the data leakage that invalidates most churn models.

**Honest result:** all three models produce near-chance AUC (~0.50), consistent with independent analyses of this synthetic source dataset — behavioural features do not encode the churn label in a recoverable way. The pipeline architecture is sound; the data is the ceiling. **NB03 segment membership is the validated churn-risk proxy for production use** — see Section 5 for the cross-validated comparison.

| Output | Contents |
|---|---|
| `churn_predictions.csv` | All customers — predicted probability and risk tier |
| `customer_risk_segments.csv` | Risk tiers cross-referenced with NB03 segment labels |
| `feature_importance.png` / `.csv` | Ranked feature contributions for the best model |
| `retention_strategies.txt` · `campaign_recommendations.txt` | Per-tier intervention playbooks |
| `churn_model.pkl` · `scaler.pkl` | Serialised model artifacts |

## 1. Setup & Configuration

`random_state = 42` seeded globally. Structured logging scoped to run ID. All window parameters, model hyperparameters, VIF thresholds, and evaluation floors live in `config.yaml → notebook4` — nothing hardcoded in the notebook or src modules.

In [1]:
# Standard imports
import sys
import logging
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Add src to path
project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(project_root / 'src'))

# Import utilities
from n4a_utils import (
    setup_logger, load_config, get_output_paths,
    print_section_header, set_run_id, get_project_root,
    validate_config
)

# ── Redirect all log handlers to stdout ──────────────────────────────────
# Ensures logger output and print() share the same stream so
# print_section_header() always appears before function log lines.
for _name in list(logging.Logger.manager.loggerDict.keys()) + ['root']:
    _lgr = logging.getLogger(_name) if _name != 'root' else logging.getLogger()
    for _h in _lgr.handlers:
        if isinstance(_h, logging.StreamHandler):
            _h.stream = sys.stdout

# Generate run ID for this execution
run_id = set_run_id()
logger = setup_logger(__name__)

# Load and validate configuration
config = load_config()
validate_config(config)   # catches missing keys early
paths = get_output_paths(config)

# Configure visualization
plt.style.use(config['visualization']['style'])
sns.set_palette(config['visualization']['color_palette'])

print(f"Pipeline Run ID: {run_id}")
print(f"Project root:    {get_project_root()}")
print(f"Output directory: {paths['figures']}")

[dafd50a1] 2026-02-28 19:25:58 - n4a_utils - INFO - Validating configuration...
[dafd50a1] 2026-02-28 19:25:58 - n4a_utils - INFO - Configuration validation passed


Pipeline Run ID: dafd50a1
Project root:    c:\Users\Rich Justine Gambe\Documents\AAA Projects\ecommerce-analytics-project
Output directory: c:\Users\Rich Justine Gambe\Documents\AAA Projects\ecommerce-analytics-project\outputs\figures\notebook4_fig


## 2. Data Loading

Loads `enhanced_df.parquet` from NB01. No modelling dataset is constructed here — that happens once in Section 2C after the feature strategy is confirmed.

**Temporal design:** the cutoff date is auto-calculated as `max_date − observation_window_days` (config: **180 days**). Features are computed from all pre-cutoff transactions; the churn label is 1 if the customer placed zero orders in the 180-day post-cutoff window.

The 180-day window was calibrated against NB03's segment recency profiles — the High-Value at Risk segment has a mean recency of 153 days. A 120-day window cuts through that distribution; 180 days provides a clean buffer above the at-risk mean and aligns with the validated segment boundary.

Exact cutoff date, customer counts, and class balance are printed by the cell below.

Note: NB01 produces 7,903 customers; NB04 models 7,671. The 232-customer difference is customers acquired after the 2025-03-15 cutoff — no pre-cutoff transaction history exists for them, so features cannot be computed. This is expected behaviour, not data loss.

In [2]:
print_section_header("DATA LOADING")

from n4b_time_split import (
    load_transaction_data, create_time_based_dataset, validate_temporal_split
)
from n4c_feature_engineering import prepare_features

# Load enhanced transaction data from NB01 output
enhanced_df = load_transaction_data(config)

# Preview temporal design — no dataset created yet.
# Strategy selection (Section 2B) determines which feature set to use.
# Final dataset is built exactly once in Section 2C after strategy is confirmed.
obs_window = config['notebook4']['observation_window_days']
# Detect date column — matches n4b_time_split._detect_date_col() logic
_date_candidates = ['order_date', 'transaction_date', 'date', 'purchase_date', 'invoice_date']
_date_col = next((c for c in _date_candidates if c in enhanced_df.columns), None)
if _date_col is None:
    raise ValueError(f"No date column found. Available: {list(enhanced_df.columns)}")

max_date    = enhanced_df[_date_col].max()
cutoff_date = max_date - __import__('datetime').timedelta(days=obs_window)

print(f"\nTemporal design:")
print(f"  Dataset range      : {enhanced_df[_date_col].min().date()} to {max_date.date()}")
print(f"  Observation window : {obs_window} days")
print(f"  Cutoff date        : {cutoff_date.date()}  (auto-calculated: max_date - {obs_window}d)")
print(f"  Feature window     : up to {cutoff_date.date()}")
print(f"  Churn observation  : {cutoff_date.date()} to {max_date.date()}")
print(f"\nSource data:")
print(f"  Transactions       : {len(enhanced_df):,}")
print(f"  Customers          : {enhanced_df['customer_id'].nunique():,}")



                                  DATA LOADING                                  



[dafd50a1] 2026-02-28 19:25:58 - n4b_time_split - INFO - ======================================================================
[dafd50a1] 2026-02-28 19:25:58 - n4b_time_split - INFO - LOADING TRANSACTION DATA
[dafd50a1] 2026-02-28 19:25:58 - n4b_time_split - INFO - ======================================================================
[dafd50a1] 2026-02-28 19:25:58 - n4b_time_split - INFO - Loading: c:\Users\Rich Justine Gambe\Documents\AAA Projects\ecommerce-analytics-project\data\processed\enhanced_df.parquet
[dafd50a1] 2026-02-28 19:25:58 - n4b_time_split - INFO - Loaded: 34,500 transactions
[dafd50a1] 2026-02-28 19:25:58 - n4b_time_split - INFO - Customers: 7,903
[dafd50a1] 2026-02-28 19:25:58 - n4b_time_split - INFO - Date range: 2023-09-12 to 2025-09-11
[dafd50a1] 2026-02-28 19:25:58 - n4b_time_split - INFO - ======================================================================



Temporal design:
  Dataset range      : 2023-09-12 to 2025-09-11
  Observation window : 180 days
  Cutoff date        : 2025-03-15  (auto-calculated: max_date - 180d)
  Feature window     : up to 2025-03-15
  Churn observation  : 2025-03-15 to 2025-09-11

Source data:
  Transactions       : 34,500
  Customers          : 7,903


## 2B. Feature Engineering: Multicollinearity Resolution

`loyalty_score` is a normalised composite of recency, frequency, and monetary (R: 30% · F: 40% · M: 30% — from `config.yaml → rfm.loyalty_weights`). Including raw RFM components alongside their composite inflates VIF without adding information. Two strategies are evaluated to resolve this:

| Strategy | Features (+ auto-added `recency_days`) | Total |
|---|---|---|
| **BASE** | Raw `frequency` + `monetary` + 5 shared features | 8 |
| **COMPOSITE** | `loyalty_score` replacing `frequency` + `monetary` + 5 shared features | 7 |

The 5 shared features across both strategies: `tenure_days`, `category_diversity`, `last_order_was_return`, `return_rate`, `discount_usage_rate`.

> `recency_days` is **always auto-prepended** by `create_time_based_dataset()` regardless of strategy — it is not a strategy choice. `all` is intentionally excluded; structural VIF inflation is guaranteed when raw components and composite are combined.

To add or remove a strategy: edit `strategy_map` in Cell 7. Feature definitions live in `n4b_time_split.get_feature_set()`.

### 2B.1 VIF Analysis

VIF threshold: **5.0** (`config.yaml → notebook4.feature_engineering.vif_threshold`). Each strategy is assessed on its own independently-built training set.

**Why BASE fails despite being conceptually simpler:**
`frequency` and `category_diversity` correlate at r = 0.846 (confirmed NB01, Section 5.4 — "frequent buyers explore more categories"). VIF detects this multivariately — `frequency` is partially predictable from the other BASE features combined, pushing its VIF above the threshold. The strict 5.0 ceiling is appropriate because Logistic Regression, the most collinearity-sensitive of the three candidates, is the model that ultimately wins.

**Why COMPOSITE passes:**
Replacing `frequency` and `monetary` with `loyalty_score` breaks the `frequency` ↔ `category_diversity` collinearity chain. Max VIF drops to 3.01 — even though `recency_days` and `loyalty_score` have a pairwise r = −0.812. Pairwise correlation and multivariate VIF are different tests.

In [3]:
print_section_header("FEATURE ENGINEERING — STRATEGY COMPARISON")

import logging
from n4c_vif_analysis import calculate_vif

# ── Log capture handler ───────────────────────────────────────────────────
# Each module logger has its own StreamHandler from setup_logger().
# We temporarily swap it out for a buffering handler, then replay
# the captured records in controlled order after our print() lines.

class _CapturingHandler(logging.Handler):
    def __init__(self):
        super().__init__()
        self.records = []
    def emit(self, record):
        self.records.append(self.format(record))
    def flush_to_stdout(self):
        for line in self.records:
            print(line)
        self.records.clear()

_capture = _CapturingHandler()
_capture.setFormatter(logging.Formatter(
    '[%(run_id)s] %(asctime)s - %(name)s - %(levelname)s - %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S'
))

_watched       = ['n4b_time_split', 'n4c_feature_engineering']
_saved_handlers = {}   # name -> list of handlers (to restore after capture)

def _attach_capture():
    """Remove existing handlers, install capture handler."""
    for name in _watched:
        lgr = logging.getLogger(name)
        _saved_handlers[name] = lgr.handlers[:]   # save
        for h in _saved_handlers[name]:
            lgr.removeHandler(h)                 # detach
        lgr.addHandler(_capture)
        lgr.propagate = False

def _detach_capture():
    """Remove capture handler, restore original handlers."""
    for name in _watched:
        lgr = logging.getLogger(name)
        lgr.removeHandler(_capture)
        for h in _saved_handlers.get(name, []):
            lgr.addHandler(h)                   
        lgr.propagate = True

# ── Strategy loop ─────────────────────────────────────────────────────────

# ── Strategy configuration ────────────────────────────────────────────────
# This map is the single control point for which strategies are compared.
# To add a new strategy:
#   1. Add an entry here:  'DISPLAY_NAME': 'config_key'
#   2. Add the config key to n4b_time_split.get_feature_set()
#   3. Define the feature list constant in n4b_time_split (e.g. NEW_FEATURES)
# Note: 'all' is intentionally excluded — structural VIF inflation guaranteed
#       (frequency + monetary + loyalty_score composite are linearly dependent).
strategy_map = {
    'BASE':      'base',
    'COMPOSITE': 'composite'
}

strategy_datasets = {}
print("Building per-strategy datasets (one create_time_based_dataset call per strategy):\n")

for display_name, strategy_key in strategy_map.items():
    print(f"[{strategy_key}] {display_name}")
    print("-" * 70)

    _attach_capture()
    try:
        X_tr_s, X_te_s, y_tr_s, y_te_s, meta_s = create_time_based_dataset(
            enhanced_df, config, feature_strategy=strategy_key
        )
        X_tr_p_s, X_te_p_s, _, _ = prepare_features(X_tr_s, X_te_s, config)
    finally:
        _detach_capture()   # always restore even if call raises

    # Replay captured logs — now guaranteed to appear after the header above
    _capture.flush_to_stdout()

    strategy_datasets[display_name] = {
        'X_train':      X_tr_p_s, 'X_test':  X_te_p_s,
        'y_train':      y_tr_s,   'y_test':  y_te_s,
        'feature_cols': meta_s['feature_cols'],
        'n_features':   len(meta_s['feature_cols'])
    }

    print(f"features ({meta_s['feature_cols']}): {meta_s['feature_cols']}\n")

# ── VIF analysis ──────────────────────────────────────────────────────────

vif_threshold = config['notebook4']['feature_engineering']['vif_threshold']
vif_results   = []

for display_name, ds in strategy_datasets.items():
    vif_data = calculate_vif(ds['X_train'], ds['feature_cols'])
    max_vif  = vif_data['VIF'].max()
    mean_vif = vif_data['VIF'].mean()
    n_high   = int((vif_data['VIF'] > vif_threshold).sum())
    vif_results.append({
        'feature_set': display_name,
        'n_features':  ds['n_features'],
        'max_vif':     round(max_vif, 2),
        'mean_vif':    round(mean_vif, 2),
        'n_high_vif':  n_high,
        'passes':      n_high == 0
    })

vif_comparison = pd.DataFrame(vif_results)

print("\nVIF ANALYSIS SUMMARY (per-strategy training set)")
print("=" * 70)
for _, row in vif_comparison.iterrows():
    status = "PASS" if row['passes'] else "FAIL"
    print(f"  {row['feature_set']:12s}: Max VIF={row['max_vif']:6.2f},  "
          f"Features > {vif_threshold}: {row['n_high_vif']:2d}  [{status}]")
print("=" * 70)


                   FEATURE ENGINEERING — STRATEGY COMPARISON                    

Building per-strategy datasets (one create_time_based_dataset call per strategy):

[base] BASE
----------------------------------------------------------------------
[dafd50a1] 2026-02-28 19:25:58 - n4b_time_split - INFO - ======================================================================
[dafd50a1] 2026-02-28 19:25:58 - n4b_time_split - INFO - CREATING TIME-BASED DATASET
[dafd50a1] 2026-02-28 19:25:58 - n4b_time_split - INFO - ======================================================================
[dafd50a1] 2026-02-28 19:25:58 - n4b_time_split - INFO - Feature lookback: 365 days before cutoff
[dafd50a1] 2026-02-28 19:25:58 - n4b_time_split - INFO - Data range: 2023-09-12 to 2025-09-11
[dafd50a1] 2026-02-28 19:25:58 - n4b_time_split - INFO - Observation window: 180 days
[dafd50a1] 2026-02-28 19:25:58 - n4b_time_split - INFO - Auto-calculated cutoff: 2025-03-15 (180d before max)
[dafd50a1] 2026-02-28 

### 2B.2 Performance Comparison

XGBoost trained independently on each strategy's dataset as a consistent strategy-selection benchmark. This is not the final model comparison — the three-way evaluation runs in Section 3.

**The expected result:** BASE will typically show higher F1 than COMPOSITE because raw `frequency` and `monetary` carry more granular signal than their composite. VIF validity gates selection — F1 is the tiebreaker only among strategies that pass. A strategy with better F1 but collinear features produces unstable coefficients for Logistic Regression, making feature importance directionally unreliable regardless of the score.

In [4]:
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
)
from n4d_model_training import train_xgboost

# ── Log capture ────────────────
class _CapturingHandler(logging.Handler):
    def __init__(self):
        super().__init__()
        self.records = []
    def emit(self, record):
        self.records.append(self.format(record))
    def flush_to_stdout(self):
        for line in self.records:
            print(line)
        self.records.clear()

_capture = _CapturingHandler()
_capture.setFormatter(logging.Formatter(
    '[%(run_id)s] %(asctime)s - %(name)s - %(levelname)s - %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S'
))

_watched        = ['n4d_model_training']
_saved_handlers = {}

def _attach_capture():
    for name in _watched:
        lgr = logging.getLogger(name)
        _saved_handlers[name] = lgr.handlers[:]
        for h in _saved_handlers[name]:
            lgr.removeHandler(h)
        lgr.addHandler(_capture)
        lgr.propagate = False

def _detach_capture():
    for name in _watched:
        lgr = logging.getLogger(name)
        lgr.removeHandler(_capture)
        for h in _saved_handlers.get(name, []):
            lgr.addHandler(h)
        lgr.propagate = True

# ── Training loop ─────────────────────────────────────────────────────────

# Train XGBoost on each strategy's independent dataset.
# XGBoost used here for strategy selection only — final model comparison
# (LR, RF, XGBoost) runs in Section 3 on the selected strategy.
strategy_results = {}

for strategy_name, ds in strategy_datasets.items():
    print(f"Training: {strategy_name} ({ds['n_features']} features)")
    print("-" * 70)

    _attach_capture()
    try:
        model_s = train_xgboost(
            ds['X_train'], ds['y_train'],
            config['notebook4']['training']['xgboost']
        )
    finally:
        _detach_capture()

    _capture.flush_to_stdout()

    y_pred_s       = model_s.predict(ds['X_test'])
    y_pred_proba_s = model_s.predict_proba(ds['X_test'])[:, 1]

    results = {
        'n_features': ds['n_features'],
        'accuracy':   accuracy_score(ds['y_test'],  y_pred_s),
        'precision':  precision_score(ds['y_test'], y_pred_s),
        'recall':     recall_score(ds['y_test'],    y_pred_s),
        'f1':         f1_score(ds['y_test'],        y_pred_s),
        'roc_auc':    roc_auc_score(ds['y_test'],   y_pred_proba_s)
    }

    strategy_results[strategy_name] = results
    print(f"\nF1-Score: {results['f1']:.4f}   ROC-AUC: {results['roc_auc']:.4f}\n")

# ── Comparison results ──────────────────────────────────────────────────────

strategy_comparison_df = pd.DataFrame(strategy_results).T.sort_values('f1', ascending=False)

print("\nPERFORMANCE COMPARISON — BASE vs COMPOSITE")
print("=" * 70)
print(strategy_comparison_df[['n_features', 'f1', 'roc_auc', 'accuracy']].to_string())
print("=" * 70)

Training: BASE (8 features)
----------------------------------------------------------------------
[dafd50a1] 2026-02-28 19:26:00 - n4d_model_training - INFO - Training XGBoost
[dafd50a1] 2026-02-28 19:26:00 - n4d_model_training - INFO - Parameters: {'n_estimators': 200, 'max_depth': 6, 'learning_rate': 0.1, 'subsample': 0.8, 'colsample_bytree': 0.8, 'random_state': 42, 'scale_pos_weight': 1.9386973180076628}
[dafd50a1] 2026-02-28 19:26:00 - n4d_model_training - INFO - Class balance: 2,088 positive / 4,048 negative
[dafd50a1] 2026-02-28 19:26:00 - n4d_model_training - INFO - Scale pos weight: 1.94
[dafd50a1] 2026-02-28 19:26:00 - n4d_model_training - INFO - Training accuracy: 0.8598

F1-Score: 0.3838   ROC-AUC: 0.5195

Training: COMPOSITE (7 features)
----------------------------------------------------------------------
[dafd50a1] 2026-02-28 19:26:00 - n4d_model_training - INFO - Training XGBoost
[dafd50a1] 2026-02-28 19:26:00 - n4d_model_training - INFO - Parameters: {'n_estimators':

### 2B.3 Decision Matrix

VIF result and XGBoost F1 combined into a single selection table.

**Selection rule:** among VIF-passing strategies, take the highest F1. If no strategy passes VIF, fall back to best F1 with a logged warning. The rule is deterministic and config-driven — no manual override required.

In [5]:
# Combine VIF and performance results into a single decision table.
# vif_comparison has 'feature_set' as a column.
# strategy_comparison_df has feature_set as INDEX.

decision_data = []

for _, vif_row in vif_comparison.iterrows():
    strategy_name = vif_row['feature_set']
    if strategy_name in strategy_comparison_df.index:
        perf_row = strategy_comparison_df.loc[strategy_name]
        decision_data.append({
            'strategy':  strategy_name,
            'features':  int(vif_row['n_features']),
            'max_vif':   float(vif_row['max_vif']),
            'vif_pass':  'PASS' if vif_row['passes'] else 'FAIL',
            'f1':        float(perf_row['f1']),
            'roc_auc':   float(perf_row['roc_auc'])
        })

decision_df = pd.DataFrame(decision_data).sort_values('f1', ascending=False)

print("FINAL DECISION MATRIX")
print("=" * 70)
print(decision_df.to_string(index=False))
print("=" * 70)


FINAL DECISION MATRIX
 strategy  features  max_vif vif_pass       f1  roc_auc
     BASE         8     7.12     FAIL 0.383784 0.519450
COMPOSITE         7     3.00     PASS 0.371930 0.505454


### 2B.4 Strategy Selection

**Selected: COMPOSITE** — the only VIF-passing strategy.

BASE fails VIF (threshold: 5.0) due to `frequency` ↔ `category_diversity` collinearity (r = 0.846, confirmed NB01). COMPOSITE passes with max VIF of 3.01.

**Trade-off acknowledged:** BASE carries a marginally higher F1, but coefficient stability gates selection. With a model already operating near chance (a property of the synthetic data, not the architecture), the small F1 gain from BASE does not justify collinear features that would make Logistic Regression coefficients — and therefore feature importance rankings — directionally unreliable.

The final dataset is rebuilt fresh with COMPOSITE in Section 2C. No data from the strategy comparison carries forward.

In [6]:
# Dynamic strategy selection — all output derived from decision_df.
vif_pass_strategies = decision_df[decision_df['vif_pass'] == 'PASS']

if len(vif_pass_strategies) > 0:
    selected = vif_pass_strategies.sort_values('f1', ascending=False).iloc[0]
    selected_strategy_key = strategy_map[selected['strategy']]
else:
    selected = decision_df.sort_values('f1', ascending=False).iloc[0]
    selected_strategy_key = strategy_map[selected['strategy']]
    print("WARNING: No strategy passes VIF. Using best F1 as fallback.")

print("=" * 80)
print(f"{'SELECTED: ' + selected['strategy']:^80}")
print("=" * 80)
print(f"  VIF Status  : {selected['vif_pass']}")
print(f"  Features    : {int(selected['features'])}")
print(f"  F1-Score    : {selected['f1']:.4f}")
print(f"  ROC-AUC     : {selected['roc_auc']:.4f}")

vif_pass_count = (decision_df['vif_pass'] == 'PASS').sum()
f1_diff = decision_df['f1'].max() - selected['f1']
print(f"\n  {vif_pass_count}/{len(decision_df)} strategies pass VIF threshold (< "
      f"{config['notebook4']['feature_engineering']['vif_threshold']})")
if f1_diff < 0.01:
    print(f"  F1 difference across strategies is negligible ({f1_diff:.4f}) — "
          "VIF validity is the tiebreaker")
else:
    print(f"  {selected['strategy']} has the best F1 among VIF-passing strategies")
print("=" * 80)

                              SELECTED: COMPOSITE                               
  VIF Status  : PASS
  Features    : 7
  F1-Score    : 0.3719
  ROC-AUC     : 0.5055

  1/2 strategies pass VIF threshold (< 5.0)
  COMPOSITE has the best F1 among VIF-passing strategies


In [7]:
print_section_header("FINAL DATASET PREPARATION")

# Build the final dataset using the selected strategy.
# This is the only create_time_based_dataset call that produces modeling data.
# Raw split — imputation intentionally deferred to prepare_features() below.
# Named _raw to clearly distinguish from the imputed+scaled _prep versions.
X_train_raw, X_test_raw, y_train, y_test, metadata = create_time_based_dataset(
    enhanced_df,
    config,
    feature_strategy=selected_strategy_key
)

# Prepare features FIRST (median imputation + StandardScaler).
# Imputer and scaler are fitted on X_train_raw only — no leakage from test set.
X_train_prep, X_test_prep, imputer, scaler = prepare_features(X_train_raw, X_test_raw, config)

# Validate AFTER imputation — Check 1 (no NaN) is correctly evaluated on
# prepared features, not raw ones. Prevents false failures from NaN fallbacks
# in compute_cutoff_features() that are legitimately handled by the imputer.
is_valid = validate_temporal_split(X_train_prep, X_test_prep, y_train, y_test, metadata, config)

if not is_valid:
    raise ValueError("Temporal split validation failed")

# Final VIF confirmation on selected feature set
from n4c_vif_analysis import check_multicollinearity

vif_final, passes_final = check_multicollinearity(
    X_train_prep,
    metadata['feature_cols'],
    threshold=config['notebook4']['feature_engineering']['vif_threshold']
)

print(f"\nFINAL DATASET — {selected['strategy']} strategy")
print("=" * 70)
print(f"  Strategy         : {metadata['feature_strategy']}")
print(f"  Features ({len(metadata['feature_cols'])})     :{metadata['feature_cols']}")
print(f"  Training samples : {len(X_train_prep):,}")
print(f"  Test samples     : {len(X_test_prep):,}")
print(f"  Churn rate       : {metadata['churn_rate']:.1%}")
print(f"  VIF status       : {'PASS' if passes_final else 'FAIL'}")
print(f"  Cutoff date      : {metadata['cutoff_date'].date()}")
print(f"  Observation window: {metadata['observation_window_days']} days")
print("=" * 70)



                           FINAL DATASET PREPARATION                            



[dafd50a1] 2026-02-28 19:26:00 - n4b_time_split - INFO - ======================================================================
[dafd50a1] 2026-02-28 19:26:00 - n4b_time_split - INFO - CREATING TIME-BASED DATASET
[dafd50a1] 2026-02-28 19:26:00 - n4b_time_split - INFO - ======================================================================
[dafd50a1] 2026-02-28 19:26:00 - n4b_time_split - INFO - Feature lookback: 365 days before cutoff
[dafd50a1] 2026-02-28 19:26:00 - n4b_time_split - INFO - Data range: 2023-09-12 to 2025-09-11
[dafd50a1] 2026-02-28 19:26:00 - n4b_time_split - INFO - Observation window: 180 days
[dafd50a1] 2026-02-28 19:26:00 - n4b_time_split - INFO - Auto-calculated cutoff: 2025-03-15 (180d before max)
[dafd50a1] 2026-02-28 19:26:00 - n4b_time_split - INFO - Cutoff: 2025-03-15
[dafd50a1] 2026-02-28 19:26:00 - n4b_time_split - INFO - Feature window: up to 2025-03-15
[dafd50a1] 2026-02-28 19:26:00 - n4b_time_split - INFO - Churn observation: 2025-03-15 to 2025-09-11
[daf


FINAL DATASET — COMPOSITE strategy
  Strategy         : composite
  Features (7)     :['recency_days', 'loyalty_score', 'tenure_days', 'category_diversity', 'last_order_was_return', 'return_rate', 'discount_usage_rate']
  Training samples : 6,136
  Test samples     : 1,535
  Churn rate       : 34.0%
  VIF status       : PASS
  Cutoff date      : 2025-03-15
  Observation window: 180 days


## 3. Model Training & Evaluation

Three classifiers trained on the COMPOSITE feature set. Class imbalance handled explicitly: `class_weight='balanced'` for Logistic Regression and Random Forest; `scale_pos_weight` derived from training split class counts for XGBoost. All hyperparameters sourced from `config.yaml → notebook4.training`.

**Selection criterion: F1-score.**
Accuracy is deliberately deprioritised — a model predicting all-active achieves high accuracy trivially on an imbalanced label. Recall is the secondary consideration: a missed churner (false negative) costs more than a wasted retention touchpoint (false positive).

In [8]:
print_section_header("MODEL TRAINING & EVALUATION")

from n4d_model_training import train_all_models
from n4e_model_evaluation import evaluate_model, compare_models, plot_roc_curves, plot_confusion_matrices

# Train all models
models = train_all_models(X_train_prep, y_train, config['notebook4']['training'])

# Evaluate each model
results_dict = {}
for model_name, model in models.items():
    results = evaluate_model(model, X_test_prep, y_test, model_name)
    results_dict[model_name] = results

# Compare models
model_comparison_df = compare_models(results_dict, config)

# Select best model
best_model_name = model_comparison_df.iloc[0]['model']
best_model      = models[best_model_name]
best_results    = results_dict[best_model_name]

print(f"\nBEST MODEL: {best_model_name}")
print("=" * 70)
print(f"F1-Score: {best_results['f1']:.4f}")
print(f"ROC-AUC:  {best_results['roc_auc']:.4f}")
print("=" * 70)


                          MODEL TRAINING & EVALUATION                           



[dafd50a1] 2026-02-28 19:26:00 - n4d_model_training - INFO - ======================================================================
[dafd50a1] 2026-02-28 19:26:00 - n4d_model_training - INFO - TRAINING ALL MODELS
[dafd50a1] 2026-02-28 19:26:00 - n4d_model_training - INFO - ======================================================================
[dafd50a1] 2026-02-28 19:26:00 - n4d_model_training - INFO - Training Logistic Regression
[dafd50a1] 2026-02-28 19:26:00 - n4d_model_training - INFO - Parameters: {'max_iter': 1000, 'random_state': 42, 'class_weight': 'balanced', 'solver': 'lbfgs'}
[dafd50a1] 2026-02-28 19:26:00 - n4d_model_training - INFO - Training accuracy: 0.5264
[dafd50a1] 2026-02-28 19:26:00 - n4d_model_training - INFO - ----------------------------------------------------------------------
[dafd50a1] 2026-02-28 19:26:00 - n4d_model_training - INFO - Training Random Forest
[dafd50a1] 2026-02-28 19:26:00 - n4d_model_training - INFO - Parameters: {'n_estimators': 200, 'max_dep


BEST MODEL: Logistic Regression
F1-Score: 0.3920
ROC-AUC:  0.4975


In [9]:
plot_roc_curves(results_dict, paths['figures'])
plot_confusion_matrices(results_dict, y_test, paths['figures'])

print("Visualizations saved to:", paths['figures'])

[dafd50a1] 2026-02-28 19:26:01 - n4e_model_evaluation - INFO - Plotting ROC curves
[dafd50a1] 2026-02-28 19:26:01 - n4e_model_evaluation - INFO - Saved: roc_curves_comparison.png
[dafd50a1] 2026-02-28 19:26:01 - n4e_model_evaluation - INFO - Plotting confusion matrices
[dafd50a1] 2026-02-28 19:26:02 - n4e_model_evaluation - INFO - Saved: confusion_matrices.png


Visualizations saved to: c:\Users\Rich Justine Gambe\Documents\AAA Projects\ecommerce-analytics-project\outputs\figures\notebook4_fig


### 3.1 Feature Importance

Importance extracted via `n4f_feature_importance.extract_feature_importance()`. For Logistic Regression: absolute coefficient magnitude (`abs(coef_[0])`), normalised to percentages — directional signal, not probability magnitude.

**Return behaviour dominates.** `return_rate` and `last_order_was_return` combined are the primary decision axis. `category_diversity` is the second independent signal.

**`loyalty_score` ranks low by construction** — it encodes recency and frequency signals already independently present in the COMPOSITE feature set. Its marginal contribution is limited because the information it summarises is not removed, just reorganised.

In [10]:
from n4f_feature_importance import extract_feature_importance, plot_feature_importance

importance_df = extract_feature_importance(
    best_model, metadata['feature_cols'], best_model_name
)
top_n_plot = min(
    config['notebook4']['feature_importance']['top_n_features'],
    len(importance_df)
)
plot_feature_importance(importance_df, paths['figures'], top_n=top_n_plot)

top_n_show = config['notebook4']['business_insights'].get('top_n_drivers', 3)

print("=" * 80)
print(f"{'FEATURE IMPORTANCE — ' + best_model_name:^80}")
print("=" * 80)
print(f"\n  {'Rank':<5} {'Feature':<28} {'Importance':>11}  {'Cumulative':>11}")
print(f"  {'-'*58}")
cumulative = 0.0
for _, row in importance_df.iterrows():
    cumulative += row['importance_pct']
    print(f"  {int(row['rank']):<5} {row['feature']:<28} {row['importance_pct']:>10.1f}%  {cumulative:>10.1f}%")

print()
top1 = importance_df.iloc[0]
print(f"  Primary driver : '{top1['feature']}' — {top1['importance_pct']:.1f}% of model decisions")

if len(importance_df) >= 2:
    top2 = importance_df.iloc[1]
    ratio = top1['importance_pct'] / top2['importance_pct']
    print(f"  Second driver  : '{top2['feature']}' — {top2['importance_pct']:.1f}% "
          f"({ratio:.1f}x less influential than primary)")

top3_cum = importance_df.head(3)['importance_pct'].sum()
print(f"  Top 3 combined : {top3_cum:.1f}% of model decisions")
print("=" * 80)

[dafd50a1] 2026-02-28 19:26:02 - n4f_feature_importance - INFO - Extracting feature importance from Logistic Regression
[dafd50a1] 2026-02-28 19:26:02 - n4f_feature_importance - INFO - Using coefficient absolute values (linear model)
[dafd50a1] 2026-02-28 19:26:02 - n4f_feature_importance - INFO - Extracted importance for 7 features
[dafd50a1] 2026-02-28 19:26:02 - n4f_feature_importance - INFO - Top 3 features:
[dafd50a1] 2026-02-28 19:26:02 - n4f_feature_importance - INFO - 1. last_order_was_return: 26.60%
[dafd50a1] 2026-02-28 19:26:02 - n4f_feature_importance - INFO - 2. return_rate: 26.05%
[dafd50a1] 2026-02-28 19:26:02 - n4f_feature_importance - INFO - 3. category_diversity: 16.19%
[dafd50a1] 2026-02-28 19:26:02 - n4f_feature_importance - INFO - Plotting top 7 features
[dafd50a1] 2026-02-28 19:26:02 - n4f_feature_importance - INFO - Saved: feature_importance.png


                    FEATURE IMPORTANCE — Logistic Regression                    

  Rank  Feature                       Importance   Cumulative
  ----------------------------------------------------------
  1     last_order_was_return              26.6%        26.6%
  2     return_rate                        26.1%        52.7%
  3     category_diversity                 16.2%        68.8%
  4     recency_days                       11.1%        79.9%
  5     discount_usage_rate                 8.3%        88.2%
  6     tenure_days                         7.6%        95.8%
  7     loyalty_score                       4.2%       100.0%

  Primary driver : 'last_order_was_return' — 26.6% of model decisions
  Second driver  : 'return_rate' — 26.1% (1.0x less influential than primary)
  Top 3 combined : 68.8% of model decisions


## 4. Business Insights & Risk Stratification

Customers are divided into **three equal-sized risk tiers** based on predicted churn probability — Low, Medium, and High Risk. Equal sizing ensures each cohort is large enough to run its own targeted retention campaign effectively. Cutoff points sourced from `config.yaml → notebook4.risk_stratification`.

> **Key finding:** A significant share of **Loyal Customers** (NB03) land in the High Risk tier. Loyal Customers are typically the highest spenders — yet they are not immune to churn. This corroborates NB03's finding that **47.1% of Loyal Customers** show at-risk or churned behaviour. Retaining this group should be a priority, not an afterthought.

In [11]:
print_section_header("BUSINESS INSIGHTS & RISK STRATIFICATION")

from n4g_risk_stratification import generate_predictions, stratify_risk, plot_risk_distribution

# Generate predictions for all customers (train + test)
X_all = metadata['full_dataset'][metadata['feature_cols']]

# monetary is computed by create_time_based_dataset for every strategy
# (part of the cutoff feature pipeline, not a feature-selection decision).
# Pull it explicitly so risk_df always has revenue data regardless of which
# strategy was selected.
_context_cols = ['monetary', 'recency_days', 'frequency']
_available    = ['customer_id', 'churn'] + [
    c for c in _context_cols if c in metadata['full_dataset'].columns
]
if 'monetary' not in metadata['full_dataset'].columns:
    logger.warning(
        "monetary not found in metadata['full_dataset'] — "
        "revenue at risk will show $0. Check n4b_time_split compute_cutoff_features()."
    )
original_df = metadata['full_dataset'][_available].copy()

# Generate predictions
predictions_df = generate_predictions(
    best_model,
    X_all,
    original_df,
    metadata['feature_cols'],
    scaler=scaler,
    imputer=imputer,   # train-only imputer — ensures full dataset uses same
                       # median statistics as model training (not all-data medians)
)

# Stratify by risk
risk_df = stratify_risk(predictions_df, config)

# Plot risk distribution
plot_risk_distribution(risk_df, paths['figures'])

print("\nRISK STRATIFICATION SUMMARY")
print("=" * 80)
# Build agg dict conditionally — avoids KeyError when monetary is absent
_agg = {
    'customer_id':       'count',
    'churn_probability': 'mean',
    'churn':             'mean',
}
if 'monetary' in risk_df.columns:
    _agg['monetary'] = 'sum'

risk_summary = risk_df.groupby('risk_level', observed=True).agg(_agg).round(3)

_cols = ['customers', 'avg_churn_prob', 'actual_churn_rate']
if 'monetary' in risk_df.columns:
    _cols.append('total_revenue')
risk_summary.columns = _cols

print(risk_summary)
print("=" * 80)


                    BUSINESS INSIGHTS & RISK STRATIFICATION                     



[dafd50a1] 2026-02-28 19:26:02 - n4g_risk_stratification - INFO - Generating predictions for all customers
[dafd50a1] 2026-02-28 19:26:02 - n4g_risk_stratification - INFO - Scaler applied to full dataset
[dafd50a1] 2026-02-28 19:26:02 - n4g_risk_stratification - INFO - Generated predictions for 7,671 customers
[dafd50a1] 2026-02-28 19:26:02 - n4g_risk_stratification - INFO - Predicted churn rate: 45.8%
[dafd50a1] 2026-02-28 19:26:02 - n4g_risk_stratification - INFO - Actual churn rate: 34.0%
[dafd50a1] 2026-02-28 19:26:02 - n4g_risk_stratification - INFO - Stratifying customers by risk level
[dafd50a1] 2026-02-28 19:26:02 - n4g_risk_stratification - INFO - Method: QUANTILE-BASED
[dafd50a1] 2026-02-28 19:26:02 - n4g_risk_stratification - INFO - Low < 0.490 (p33%)
[dafd50a1] 2026-02-28 19:26:02 - n4g_risk_stratification - INFO - High >= 0.506 (p67%)
[dafd50a1] 2026-02-28 19:26:02 - n4g_risk_stratification - INFO - Risk distribution:
[dafd50a1] 2026-02-28 19:26:02 - n4g_risk_stratificatio


RISK STRATIFICATION SUMMARY
            customers  avg_churn_prob  actual_churn_rate  total_revenue
risk_level                                                             
Low              2532           0.477              0.324   889361.62500
Medium           2607           0.498              0.337   708478.93750
High             2532           0.524              0.359   436550.71875


### 4.1 Business Recommendations

Tiered by urgency and cost-efficiency. Discount percentages, response targets, and retention budgets sourced from `config.yaml → notebook4.business_insights.campaigns`.

**🔴 High Risk — Immediate multi-channel outreach** (email + phone + SMS).
Within this tier, Loyal Customers (NB03) warrant the highest retention spend per customer given their monetary profile and the cost of losing them.

**🟡 Medium Risk — Proactive engagement** (email + SMS).
Pair with a short friction survey — identifying the pain point before churn converts is cheaper than recovery after the fact.

**🟢 Low Risk — Regular communications and loyalty rewards.**
Avoid over-messaging — contact fatigue is itself a churn accelerant for customers who are not yet at risk.

**Cross-tier lever:** `return_rate` and `last_order_was_return` are the primary model signals across all tiers. Proactive post-return outreach — particularly for discount-dependent customers — has the highest leverage regardless of tier. Discounts that train price sensitivity rather than build loyalty are a structural churn risk.

In [12]:
from n4h_business_insights import (
    generate_retention_strategies,
    create_campaign_recommendations,
    load_segment_names
)

model_metrics = {
    'accuracy':  best_results['accuracy'],
    'precision': best_results['precision'],
    'recall':    best_results['recall'],
    'f1':        best_results['f1'],
    'roc_auc':   best_results['roc_auc']
}

strategies = generate_retention_strategies(
    importance_df, risk_df, model_metrics, config
)
campaigns = create_campaign_recommendations(risk_df, config)

print(strategies)
print(campaigns)

# Executive summary — top drivers + risk distribution
print("=" * 80)
print(f"{'EXECUTIVE SUMMARY':^80}")
print("=" * 80)
print(f"  Best model   : {best_model_name}  |  F1: {best_results['f1']:.3f}  |  ROC-AUC: {best_results['roc_auc']:.3f}")

top_n = config['notebook4']['business_insights'].get('top_n_drivers', 3)
print(f"\n  Top {top_n} churn drivers:")
for _, row in importance_df.head(top_n).iterrows():
    print(f"    {int(row['rank'])}. {row['feature']:<28} {row['importance_pct']:.1f}%")

return_features = ['return_rate', 'last_order_was_return']
return_pct = importance_df[importance_df['feature'].isin(return_features)]['importance_pct'].sum()
if return_pct > 0:
    print(f"\n  Return behavior combined : {return_pct:.1f}% of model decisions")

print(f"\n  Risk distribution:")
for level in config['notebook4']['risk_stratification']['risk_levels']:
    count = (risk_df['risk_level'] == level).sum()
    pct   = count / len(risk_df) * 100
    print(f"    {level:<8} {count:>6,} customers  ({pct:.1f}%)")
print("=" * 80)


[dafd50a1] 2026-02-28 19:26:03 - n4h_business_insights - INFO - Generating retention strategies
[dafd50a1] 2026-02-28 19:26:03 - n4h_business_insights - INFO - Loaded 7,903 segment assignments from NB03
[dafd50a1] 2026-02-28 19:26:03 - n4h_business_insights - INFO - Segments: {'Needs Engagement': 3766, 'Loyal Customers': 2215, 'Lost Customers': 1442, 'High-Value at Risk': 480}
[dafd50a1] 2026-02-28 19:26:03 - n4h_business_insights - INFO - Retention strategies generated
[dafd50a1] 2026-02-28 19:26:03 - n4h_business_insights - INFO - Creating campaign recommendations
[dafd50a1] 2026-02-28 19:26:03 - n4h_business_insights - INFO - Loaded 7,903 segment assignments from NB03
[dafd50a1] 2026-02-28 19:26:03 - n4h_business_insights - INFO - Segments: {'Needs Engagement': 3766, 'Loyal Customers': 2215, 'Lost Customers': 1442, 'High-Value at Risk': 480}
[dafd50a1] 2026-02-28 19:26:03 - n4h_business_insights - INFO - Campaign recommendations created


CHURN RETENTION STRATEGIES

MODEL PERFORMANCE
--------------------------------------------------------------------------------
Accuracy:  52.3%
Precision: 34.6%
Recall:    45.2%
F1-Score:  39.2%
ROC-AUC:   0.498

TOP CHURN DRIVERS
--------------------------------------------------------------------------------
1. last_order_was_return           26.60%
2. return_rate                     26.05%
3. category_diversity              16.19%
4. recency_days                    11.10%
5. discount_usage_rate              8.28%

NB03 SEGMENT x CHURN RISK BREAKDOWN
--------------------------------------------------------------------------------
  Cross-notebook finding: how NB03 segments distribute across churn risk tiers.

risk_level           Low  Medium  High  Total  High_Risk_Pct
segment_name                                                
High-Value at Risk   159     153   156    468           33.3
Lost Customers       163     490   789   1442           54.7
Loyal Customers     1235     668   

In [13]:
print_section_header("EXPORT DELIVERABLES")

from n4i_export import (
    export_predictions,
    export_business_deliverables,
    export_model_artifacts
)

processed_dir = get_project_root() / config['paths']['processed_data']
models_dir    = get_project_root() / config['paths']['models_dir']

# Export customer predictions
predictions_path = export_predictions(
    risk_df,
    processed_dir,
    'churn_predictions.csv'
)

# Export business deliverables
deliverable_paths = export_business_deliverables(
    importance_df,
    risk_df,
    strategies,
    campaigns,
    processed_dir
)

# Export model artifacts for deployment
artifact_paths = export_model_artifacts(
    best_model,
    scaler,
    imputer,
    metadata['feature_cols'],
    models_dir,
    model_performance=best_results
)

print(f"\nPredictions:  {predictions_path}")
print(f"\nBusiness deliverables ({len(deliverable_paths)} files):")
for key, path in deliverable_paths.items():
    print(f"  {key}: {path.name}")
print(f"\nModel artifacts ({len(artifact_paths)} files):")
for key, path in artifact_paths.items():
    print(f"  {key}: {path.name}")
print("\nExport complete.")


                              EXPORT DELIVERABLES                               



[dafd50a1] 2026-02-28 19:26:03 - n4i_export - INFO - Exporting predictions
[dafd50a1] 2026-02-28 19:26:03 - n4i_export - INFO -   Saved: churn_predictions.csv
[dafd50a1] 2026-02-28 19:26:03 - n4i_export - INFO -   Total customers: 7,671
[dafd50a1] 2026-02-28 19:26:03 - n4i_export - INFO -   Predicted churn: 3,514 (45.8%)
[dafd50a1] 2026-02-28 19:26:03 - n4i_export - INFO - Exporting business deliverables
[dafd50a1] 2026-02-28 19:26:03 - n4i_export - INFO -   Saved: feature_importance.csv
[dafd50a1] 2026-02-28 19:26:03 - n4i_export - INFO -   Saved: customer_risk_segments.csv
[dafd50a1] 2026-02-28 19:26:03 - n4i_export - INFO -   Saved: retention_strategies.txt
[dafd50a1] 2026-02-28 19:26:03 - n4i_export - INFO -   Saved: campaign_recommendations.txt
[dafd50a1] 2026-02-28 19:26:03 - n4i_export - INFO - Exported 4 deliverables
[dafd50a1] 2026-02-28 19:26:03 - n4i_export - INFO - Exporting model artifacts
[dafd50a1] 2026-02-28 19:26:03 - n4i_export - INFO -   Saved: churn_model.pkl
[dafd5


Predictions:  c:\Users\Rich Justine Gambe\Documents\AAA Projects\ecommerce-analytics-project\data\processed\churn_predictions.csv

Business deliverables (4 files):
  importance: feature_importance.csv
  risk_segments: customer_risk_segments.csv
  strategies: retention_strategies.txt
  campaigns: campaign_recommendations.txt

Model artifacts (4 files):
  model: churn_model.pkl
  scaler: scaler.pkl
  features: feature_columns.txt
  performance: model_performance.json

Export complete.


## 5. Data Quality Assessment & Model Limitations

All five evaluation metrics fail the configured thresholds. **ROC-AUC sits at chance (~0.50).**

**This is not a pipeline failure — it is a data ceiling.**
Independent analyses of this synthetic source dataset consistently return AUC of 0.52–0.53. The churn label is not recoverable from behavioural features in synthetically generated data; it is a property of the data generator. The temporal isolation architecture is methodologically correct.

**Validated fallback: NB03 segment membership as churn proxy.**

When k-means segments from NB03 are cross-validated against NB04's time-based churn labels, they produce a clean, monotonic churn gradient that is far more actionable than near-random probability scores:

| NB03 Segment | Churn Rate (NB04 labels) | Signal quality |
|---|---|---|
| Lost Customers | 94.3% | Definitive |
| High-Value at Risk | 32.3% | Strong |
| Needs Engagement | 25.8% | Moderate |
| Loyal Customers | 8.2% | Low risk |

Rates are slightly lower than NB03's recency-based churn flags for the same segments — a subset of "churned by recency" customers do return within the 180-day observation window.

**Production recommendation:** use NB03 segment membership as the primary churn risk signal. This notebook's three-tier stratification adds value for campaign sizing and channel allocation, but segment label should drive prioritisation.

In [14]:
# Dynamic data quality assessment — all values from best_results and config.

thresholds = config['notebook4']['evaluation']['thresholds']
roc = best_results['roc_auc']

print("=" * 70)
print(f"{'DATA QUALITY ASSESSMENT':^70}")
print("=" * 70)
print(f"  {'Metric':<14} {'Value':>8}  {'Min':>8}  {'Status'}")
print(f"  {'-'*44}")

all_pass = True
for metric, min_key in [
    ('accuracy',  'min_accuracy'),
    ('precision', 'min_precision'),
    ('recall',    'min_recall'),
    ('f1',        'min_f1'),
    ('roc_auc',   'min_roc_auc'),
]:
    val   = best_results[metric]
    floor = thresholds[min_key]
    flag  = "PASS" if val >= floor else "FAIL"
    if val < floor:
        all_pass = False
    print(f"  {metric:<14} {val:>8.3f}  {floor:>8.3f}  [{flag}]")

print()
if all_pass:
    print("  All thresholds met — model is production-ready.")
else:
    print("  Thresholds not met — synthetic data ceiling confirmed.")
    if roc < 0.60:
        print(f"  ROC-AUC = {roc:.3f} (chance level).")
        print("  Independent Kaggle analysis: consistent upper bound AUC 0.52-0.53.")
        print("  Source: kaggle.com/datasets/miadul/e-commerce-sales-transactions-dataset/discussion/610656")
        print("\n  Recommendation: use NB03 segment labels as production churn risk signal.")
        try:
            segs = load_segment_names(config)
            if segs is not None and 'churn' in risk_df.columns:
                merged = risk_df.merge(segs, on='customer_id', how='left')
                rates  = merged.groupby('segment_name')['churn'].mean().sort_values(ascending=False)
                print(f"\n  {'Segment':<30} {'Actual Churn Rate':>18}")
                print(f"  {'-'*50}")
                for seg, rate in rates.items():
                    print(f"  {str(seg):<30} {rate:>17.1%}")
        except Exception:
            pass

print("=" * 70)


[dafd50a1] 2026-02-28 19:26:03 - n4h_business_insights - INFO - Loaded 7,903 segment assignments from NB03
[dafd50a1] 2026-02-28 19:26:03 - n4h_business_insights - INFO - Segments: {'Needs Engagement': 3766, 'Loyal Customers': 2215, 'Lost Customers': 1442, 'High-Value at Risk': 480}


                       DATA QUALITY ASSESSMENT                        
  Metric            Value       Min  Status
  --------------------------------------------
  accuracy          0.523     0.750  [FAIL]
  precision         0.346     0.700  [FAIL]
  recall            0.452     0.700  [FAIL]
  f1                0.392     0.700  [FAIL]
  roc_auc           0.498     0.800  [FAIL]

  Thresholds not met — synthetic data ceiling confirmed.
  ROC-AUC = 0.498 (chance level).
  Independent Kaggle analysis: consistent upper bound AUC 0.52-0.53.
  Source: kaggle.com/datasets/miadul/e-commerce-sales-transactions-dataset/discussion/610656

  Recommendation: use NB03 segment labels as production churn risk signal.

  Segment                         Actual Churn Rate
  --------------------------------------------------
  Lost Customers                             94.3%
  High-Value at Risk                         32.3%
  Needs Engagement                           25.8%
  Loyal Customers            

## Final Summary

**Notebook 04 complete.**

| Stage | Result |
|---|---|
| Customers | 7,671 |
| Feature strategy | COMPOSITE — only VIF-passing strategy (max VIF: 3.01) |
| Best model | Logistic Regression (selected by F1) |
| ROC-AUC | ~0.50 — near chance; synthetic data ceiling confirmed |
| Risk tiers | Three equal-sized cohorts: Low · Medium · High |
| Primary churn signal | NB03 segment membership (validated proxy) |

**Architectural note:** `n4b_data_loader.py` is present in `src/` but disabled — a development artifact that derives the churn label from a static RFM snapshot rather than forward-looking transaction timestamps. Preserved as a design decision record; do not re-enable without replacing the churn definition.

Load `customer_risk_segments.csv` in NB05 for cross-notebook segment × fraud analysis.

In [15]:
# Final summary — all values dynamically pulled from metadata, selected strategy, and best model results.

print("=" * 80)
print(f"{'NOTEBOOK 04 — FINAL SUMMARY':^80}")
print("=" * 80)

# Pipeline parameters
print(f"\n  {'Cutoff date':<22}: {metadata['cutoff_date'].date()}")
print(f"  {'Observation window':<22}: {metadata['observation_window_days']} days")
print(f"  {'Customers modeled':<22}: {metadata['n_customers']:,}  (churn rate: {metadata['churn_rate']:.1%})")
n_train, n_test = len(X_train_prep), len(X_test_prep)
print(f"  {'Train / Test':<22}: {n_train:,} / {n_test:,}  ({n_train/(n_train+n_test):.0%}/{n_test/(n_train+n_test):.0%} split)")

# Strategy & model
print(f"\n  {'Strategy selected':<22}: {selected['strategy']}  ({int(selected['features'])} features, VIF {selected['vif_pass']})")
print(f"  {'Best model':<22}: {best_model_name}")
print(f"  {'F1 / ROC-AUC':<22}: {best_results['f1']:.3f} / {best_results['roc_auc']:.3f}")
print(f"  {'Recall':<22}: {best_results['recall']:.3f}")

# Feature importance
print(f"\n  {'Top churn drivers:':<22}\n")
cumulative = 0.0
for _, row in importance_df.iterrows():
    cumulative += row['importance_pct']
    print(f"  {int(row['rank'])}.  {row['feature']:<26} {row['importance_pct']:>5.1f}%  (cum {cumulative:.1f}%)")

# Risk tiers
print(f"\n  {'Risk stratification:':<22}\n")
for level in reversed(config['notebook4']['risk_stratification']['risk_levels']):
    mask  = risk_df['risk_level'] == level
    count = mask.sum()
    avg_p = risk_df.loc[mask, 'churn_probability'].mean()
    rev   = risk_df.loc[mask, 'monetary'].sum() if 'monetary' in risk_df.columns else 0
    print(f"  {level:<8} {count:>6,} customers  avg prob {avg_p:.2f}  revenue at risk ${rev:>12,.0f}")

print(f"\n  Deliverables exported to: {get_project_root() / config['paths']['processed_data']}")
print(f"  Run ID: {run_id}")
print("=" * 80)


                          NOTEBOOK 04 — FINAL SUMMARY                           

  Cutoff date           : 2025-03-15
  Observation window    : 180 days
  Customers modeled     : 7,671  (churn rate: 34.0%)
  Train / Test          : 6,136 / 1,535  (80%/20% split)

  Strategy selected     : COMPOSITE  (7 features, VIF PASS)
  Best model            : Logistic Regression
  F1 / ROC-AUC          : 0.392 / 0.498
  Recall                : 0.452

  Top churn drivers:    

  1.  last_order_was_return       26.6%  (cum 26.6%)
  2.  return_rate                 26.1%  (cum 52.7%)
  3.  category_diversity          16.2%  (cum 68.8%)
  4.  recency_days                11.1%  (cum 79.9%)
  5.  discount_usage_rate          8.3%  (cum 88.2%)
  6.  tenure_days                  7.6%  (cum 95.8%)
  7.  loyalty_score                4.2%  (cum 100.0%)

  Risk stratification:  

  High      2,532 customers  avg prob 0.52  revenue at risk $     436,551
  Medium    2,607 customers  avg prob 0.50  revenue at ri